# AFib detector — leave-one-patient-out evaluation (Notebook 2)
**Bektur Asilov · COMSE-23 · Ala-Too**

Notebook 1 used one fixed split (5 test patients). Here every one of the 23 MIT-BIH AFDB patients is the test patient once:
the RR-interval TCN (same architecture as the firmware model, 5,665 parameters) is trained on the other 22 and tested on the one left out.

Reported, without any tuning on the test patient (threshold fixed at 0.5):
1. **Window level** — sequences of 32 beats fully inside one rhythm episode (same as Notebook 1), float32 and int8.
2. **Continuous timeline** — a decision every 4 beats over the *whole* recording, as on the device (transitions and noise included).
3. **2-of-3 smoothing** — an alarm needs 2 positive decisions out of the last 3.
4. **Episode level** — AF episodes ≥ 30 s: share detected, detection delay; **false alarms per hour** of normal rhythm; AF-burden error.

> Kaggle: *Add Input* → `marufhs/atrial-fibrillation`; Internet On (pip install wfdb). CPU is enough (~1 h).

In [ ]:
import os, glob, time, random, contextlib, io, json, subprocess, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score

SEED = 42
FS_ORIG, FS = 250, 125
N_RR, RR_STRIDE, DEV_STRIDE = 32, 8, 4
MIN_EP_S = 30
OUT_DIR = '/kaggle/working'
LABELS = {'(N': 0, '(AFIB': 1}
tf.get_logger().setLevel('ERROR')

DB_DIR = [d for d in {os.path.dirname(p) for p in glob.glob('/kaggle/input/**/04015.dat', recursive=True)} if '/old' not in d][0]
records = sorted(r[:-4] for r in os.listdir(DB_DIR) if r.endswith('.dat') and os.path.exists(os.path.join(DB_DIR, r[:-4] + '.atr')))
print(len(records), 'records:', records)

def load(rec):
    sig = np.nan_to_num(wfdb.rdrecord(os.path.join(DB_DIR, rec), channels=[0]).p_signal[:, 0])
    ann = wfdb.rdann(os.path.join(DB_DIR, rec), 'atr')
    starts = list(ann.sample); ends = starts[1:] + [len(sig)]
    return sig, [(s, e, LABELS.get(a.strip().rstrip('\x00'))) for s, e, a in zip(starts, ends, ann.aux_note)]
DATA = {r: load(r) for r in records}

## R-peak detector (identical to the firmware) and RR features

In [ ]:
SOS = butter(2, [0.5, 40], btype='band', fs=FS_ORIG, output='sos')
MA = 19; DECAY = 0.5 ** (1 / (2.0 * FS)); THR, REFR, WARM = 0.35, int(0.25 * FS), 2 * FS
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

@njit
def detect_r(x):
    out = np.empty(len(x) // 20 + 10, np.int64); n = 0
    buf = np.zeros(MA); ssum = 0.0; prev = x[0]; pk = 0.0; last = -10**9; armed = True
    for i in range(len(x)):
        d = x[i] - prev; prev = x[i]; e = d * d
        ssum += e - buf[i % MA]; buf[i % MA] = e
        env = ssum / MA; pk = max(pk * DECAY, env)
        if i < WARM: continue
        if armed and env > THR * pk and i - last > REFR:
            out[n] = i; n += 1; last = i; armed = False
            if n >= len(out): break
        elif not armed and env < 0.5 * THR * pk:
            armed = True
    return out[:n]

def rr_features(rr):
    rr = np.clip(rr, 0.2, 2.0)
    return np.stack([(rr - 0.8) / 0.25, (rr / rr.mean() - 1) / 0.15], -1).astype(np.float32)

PREP = {}
t0 = time.time()
for rec in records:
    sig, segs = DATA[rec]
    x = sosfilt(SOS, sig)[::2]
    beats = detect_r(x); rr = np.diff(beats) / FS
    lab_at = np.full(len(x), -1, np.int8)
    for s, e, l in segs:
        if l is not None: lab_at[s // 2:e // 2] = l
    # (a) clean windows (Notebook 1 protocol)
    Xw, yw = [], []
    for k in range(0, len(rr) - N_RR + 1, RR_STRIDE):
        labs = lab_at[beats[k]:beats[k + N_RR]]
        if labs.min() < 0 or labs.min() != labs.max(): continue
        Xw.append(rr_features(rr[k:k + N_RR])); yw.append(int(labs[0]))
    # (b) device timeline: a decision every 4 beats, label = rhythm at the decision moment
    Xt, yt, tt = [], [], []
    for k in range(0, len(rr) - N_RR + 1, DEV_STRIDE):
        end = beats[k + N_RR]
        Xt.append(rr_features(rr[k:k + N_RR])); yt.append(int(lab_at[min(end, len(lab_at) - 1)])); tt.append(end / FS)
    eps = [(s / FS_ORIG, e / FS_ORIG) for s, e, l in segs if l == 1 and (e - s) / FS_ORIG >= MIN_EP_S]
    n_sec = sum(e - s for s, e, l in segs if l == 0) / FS_ORIG
    a_sec = sum(e - s for s, e, l in segs if l == 1) / FS_ORIG
    PREP[rec] = dict(Xw=np.array(Xw, np.float32).reshape(-1, N_RR, 2), yw=np.array(yw, np.int32),
                     Xt=np.array(Xt, np.float32).reshape(-1, N_RR, 2), yt=np.array(yt, np.int8), tt=np.array(tt),
                     eps=eps, n_h=n_sec / 3600, af_h=a_sec / 3600)
print(f'prepared in {time.time()-t0:.0f} s')
display(pd.DataFrame([dict(record=r, N_hours=round(p['n_h'], 2), AF_hours=round(p['af_h'], 2),
                           AF_episodes_30s=len(p['eps']), windows=len(p['yw'])) for r, p in PREP.items()]))

## Model and helpers (unchanged from Notebook 1)

In [ ]:
def tcn_block(x, f, k, d, drop):
    sc = x
    x = layers.Conv1D(f, k, padding='causal', dilation_rate=d, activation='relu')(x)
    x = layers.Dropout(drop)(x)
    x = layers.Conv1D(f, k, padding='causal', dilation_rate=d)(x)
    if sc.shape[-1] != f: sc = layers.Conv1D(f, 1)(sc)
    return layers.ReLU()(layers.Add()([x, sc]))

def build_rr_tcn():
    inp = keras.Input(shape=(N_RR, 2)); x = inp
    for d in (1, 2, 4, 8): x = tcn_block(x, 16, 3, d, 0.1)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(layers.GlobalAveragePooling1D()(x)))

def to_int8(m, Xrep):
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    rep = Xrep[np.random.RandomState(0).choice(len(Xrep), min(1000, len(Xrep)), replace=False)]
    conv.representative_dataset = lambda: ([s[None]] for s in rep)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return conv.convert()

def run_tflite(blob, Xd, bs=2048):
    outs = []
    for i in range(0, len(Xd), bs):
        xb = Xd[i:i + bs]
        it = tf.lite.Interpreter(model_content=blob); inp = it.get_input_details()[0]
        it.resize_tensor_input(inp['index'], [len(xb), N_RR, 2]); it.allocate_tensors()
        inp, out = it.get_input_details()[0], it.get_output_details()[0]
        sc, zp = inp['quantization']; it.set_tensor(inp['index'], np.clip(np.round(xb / sc + zp), -128, 127).astype(np.int8))
        it.invoke(); sc, zp = out['quantization']
        outs.append(((it.get_tensor(out['index']).astype(np.float32) - zp) * sc).ravel())
    return np.concatenate(outs) if outs else np.zeros(0)

def cm(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel(); return tn, fp, fn, tp

def smooth_2of3(pred):
    p = pred.astype(int); s = p.copy()
    s[1:] += p[:-1]; s[2:] += p[:-2]
    return (s >= 2).astype(int)

def episode_eval(tt, alarm, yt, eps):
    det, delays = 0, []
    for s, e in eps:
        m = (tt >= s) & (tt <= e) & (alarm == 1)
        if m.any(): det += 1; delays.append(tt[m][0] - s)
    onsets = np.where(np.diff(np.r_[0, alarm]) == 1)[0]
    fa = int(sum(yt[i] == 0 for i in onsets))
    return det, delays, fa

## Leave-one-patient-out loop (23 folds)

In [ ]:
rows, pooled = [], {k: [] for k in ('yw', 'pw', 'pw8', 'yt', 'pt8', 'st8')}
T0 = time.time()
for fi, test in enumerate(records):
    tf.keras.backend.clear_session(); random.seed(SEED + fi); np.random.seed(SEED + fi); tf.random.set_seed(SEED + fi)
    rest = [r for r in records if r != test]
    rng = np.random.RandomState(SEED + fi)
    for _ in range(500):   # 2 validation patients (for early stopping only) with both rhythms present
        val = list(rng.choice(rest, 2, replace=False))
        if all(sum((PREP[v]['yw'] == c).sum() for v in val) >= 100 for c in (0, 1)): break
    train = [r for r in rest if r not in val]
    Xtr = np.concatenate([PREP[r]['Xw'] for r in train]); ytr = np.concatenate([PREP[r]['yw'] for r in train])
    Xva = np.concatenate([PREP[r]['Xw'] for r in val]);   yva = np.concatenate([PREP[r]['yw'] for r in val])
    cw = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * ytr.sum())}
    m = build_rr_tcn()
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).shuffle(50000, seed=SEED).batch(256).prefetch(2)
    h = m.fit(ds, validation_data=(Xva, yva), epochs=40, class_weight=cw, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)])
    blob = to_int8(m, Xtr)
    P = PREP[test]
    pw = m.predict(P['Xw'], batch_size=2048, verbose=0).ravel() if len(P['yw']) else np.zeros(0)
    pw8 = run_tflite(blob, P['Xw']); pt8 = run_tflite(blob, P['Xt'])
    yw = P['yw']; yt = P['yt']; keep = yt >= 0
    raw8 = (pt8 >= 0.5).astype(int); st8 = smooth_2of3(raw8)
    det, delays, fa = episode_eval(P['tt'], st8, yt, P['eps'])
    det_raw, _, fa_raw = episode_eval(P['tt'], raw8, yt, P['eps'])
    tn, fp, fn, tp = cm(yw, (pw8 >= 0.5).astype(int))
    tn2, fp2, fn2, tp2 = cm(yt[keep], st8[keep])
    est_burden = st8[keep].mean() * 100; true_burden = (yt[keep] == 1).mean() * 100
    rows.append(dict(record=test, val='+'.join(val), epochs=len(h.history['loss']),
                     N_h=P['n_h'], AF_h=P['af_h'], n_win=len(yw), af_win=int(yw.sum()),
                     win_sens_f32=((pw >= .5) & (yw == 1)).sum() / max(1, yw.sum()) if yw.sum() else np.nan,
                     win_sens_i8=tp / (tp + fn) if tp + fn else np.nan, win_spec_i8=tn / (tn + fp) if tn + fp else np.nan,
                     tl_sens_s=tp2 / (tp2 + fn2) if tp2 + fn2 else np.nan, tl_spec_s=tn2 / (tn2 + fp2) if tn2 + fp2 else np.nan,
                     ep_total=len(P['eps']), ep_det=det, ep_det_raw=det_raw,
                     delay_med_s=np.median(delays) if delays else np.nan,
                     fa=fa, fa_raw=fa_raw, true_burden=true_burden, est_burden=est_burden))
    pooled['yw'].append(yw); pooled['pw'].append(pw); pooled['pw8'].append(pw8)
    pooled['yt'].append(yt[keep]); pooled['pt8'].append(pt8[keep]); pooled['st8'].append(st8[keep])
    r = rows[-1]
    print(f"[{fi+1:2d}/{len(records)}] {test}  ep={r['epochs']:2d}  win sens {r['win_sens_i8']:.3f} spec {r['win_spec_i8']:.3f}  "
          f"episodes {det}/{len(P['eps'])}  FA {fa} ({P['n_h']:.1f} h N)  burden {true_burden:.1f}->{est_burden:.1f}%  "
          f"[{time.time()-T0:.0f} s]", flush=True)
F = pd.DataFrame(rows); F.to_csv(f'{OUT_DIR}/lopo_folds.csv', index=False)

## Results

In [ ]:
cat = {k: np.concatenate(v) for k, v in pooled.items()}
def summ(y, p, thr=.5, prob=None):
    tn, fp, fn, tp = cm(y, (p >= thr).astype(int) if prob is None else p)
    d = dict(sensitivity=tp / (tp + fn), specificity=tn / (tn + fp), accuracy=(tp + tn) / len(y), F1=2 * tp / (2 * tp + fp + fn))
    if prob is None: d['ROC_AUC'] = roc_auc_score(y, p)
    return d
Nh = F.N_h.sum()
S = {
  'window_float32_pooled': summ(cat['yw'], cat['pw']),
  'window_int8_pooled': summ(cat['yw'], cat['pw8']),
  'timeline_int8_raw': summ(cat['yt'], (cat['pt8'] >= .5).astype(int), prob=1),
  'timeline_int8_2of3': summ(cat['yt'], cat['st8'], prob=1),
  'window_int8_per_patient_mean': dict(sensitivity=F.win_sens_i8.mean(), specificity=F.win_spec_i8.mean(),
                                       sens_median=F.win_sens_i8.median(), sens_min=F.win_sens_i8.min()),
  'episodes_30s': dict(total=int(F.ep_total.sum()), detected_2of3=int(F.ep_det.sum()), detected_raw=int(F.ep_det_raw.sum()),
                       sensitivity_2of3=F.ep_det.sum() / F.ep_total.sum(), median_delay_s=float(np.nanmedian(F.delay_med_s))),
  'false_alarms': dict(normal_hours=Nh, fa_2of3=int(F.fa.sum()), fa_raw=int(F.fa_raw.sum()),
                       per_hour_2of3=F.fa.sum() / Nh, per_hour_raw=F.fa_raw.sum() / Nh, per_24h_2of3=24 * F.fa.sum() / Nh),
  'burden_abs_error_pp': dict(mean=float((F.est_burden - F.true_burden).abs().mean()),
                              median=float((F.est_burden - F.true_burden).abs().median())),
  'patients': len(F), 'with_AF_windows': int((F.af_win > 0).sum()),
}
json.dump(S, open(f'{OUT_DIR}/lopo_summary.json', 'w'), indent=2, default=float)
print(json.dumps(S, indent=2, default=lambda v: round(float(v), 4)))
display(F.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
G = F.sort_values('win_sens_i8')
ax[0].bar(G.record, G.win_sens_i8 * 100, color='#C0362F', label='sensitivity')
ax[0].plot(G.record, G.win_spec_i8 * 100, 'ko', label='specificity')
ax[0].axhline(S['window_int8_pooled']['sensitivity'] * 100, ls='--', c='gray'); ax[0].set_ylim(0, 102)
ax[0].set_title('LOPO, int8, per test patient'); ax[0].tick_params(axis='x', rotation=70); ax[0].legend()
ax[1].scatter(F.true_burden, F.est_burden, c='#C0362F'); ax[1].plot([0, 100], [0, 100], 'k--', lw=1)
ax[1].set_xlabel('true AF burden, %'); ax[1].set_ylabel('estimated (2-of-3), %'); ax[1].set_title('AF burden per patient')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/lopo_per_patient.png', dpi=150); plt.show()
print('total time', round((time.time() - T0) / 60, 1), 'min')